In [ ]:
# ============================================================
#  CLASE: WORKING WITH LLMs - API DE DEEPSEEK (INTERACTIVO)
#  Caso práctico: Análisis de tickets de soporte
# ============================================================

from openai import OpenAI
import json
import time
import getpass

# ============================================================
#  1. CLIENTE Y AUTENTICACIÓN
# ============================================================
# La API_KEY se obtiene en https://platform.deepseek.com
# Usamos getpass para que la clave no se vea en pantalla.

print("=" * 60)
print("  CONFIGURACIÓN INICIAL - API DE DEEPSEEK")
print("=" * 60)

api_key = getpass.getpass("🔑 Ingresa tu API KEY de DeepSeek: ")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

print("✅ Cliente configurado correctamente.\n")


In [ ]:

# ============================================================
#  DATOS DE EJEMPLO (precargados, así no se tipean a mano)
# ============================================================
TICKETS = {
    1: """Compré la licencia anual hace 3 meses y desde la última actualización
la app se cierra sola al exportar reportes. Tengo entrega mañana al directorio.
Ya reinstalé y nada. Quiero solución o reembolso completo.""",

    2: """Hola, quería saber si la suscripción Pro incluye el módulo de
analítica avanzada o si se paga aparte. También me gustaría saber si hay
descuento por pago anual. Gracias.""",

    3: """LLEVO 4 DÍAS SIN PODER ENTRAR A MI CUENTA. He escrito 3 veces
y nadie responde. Esto es INACEPTABLE, tengo un equipo de 12 personas
parado por su culpa. Exijo hablar con un supervisor YA.""",

    4: """Buenas tardes. Notamos que en el dashboard los gráficos de
ingresos del mes muestran cifras distintas a las del reporte exportado
en PDF. ¿Podrían revisarlo? Adjunto capturas en otro mail.""",
}

ROLES = {
    1: "Eres un analista de soporte técnico empático, conciso y propones 2 acciones concretas.",
    2: "Eres un agente de ventas amable. Identificas oportunidades y respondes con claridad comercial.",
    3: "Eres un clasificador de tickets. Solo devuelves JSON válido, nada de texto extra.",
}


# ============================================================
#  MENÚ PRINCIPAL
# ============================================================
def mostrar_menu():
    print("=" * 60)
    print("  ¿QUÉ QUIERES PROBAR?")
    print("=" * 60)
    print("  1. Llamada básica (system prompt + temperature + max_tokens)")
    print("  2. Comparar TEMPERATURE: 0 vs 0.7 vs 1.5")
    print("  3. Salida estructurada en JSON")
    print("  4. Conversación multi-turno")
    print("  5. Streaming (respuesta token a token)")
    print("  6. Function calling (tool use)")
    print("  7. Alucinación: con y sin mitigación")
    print("  0. Salir")
    return input("\nElige opción: ").strip()


def elegir_ticket():
    print("\n📋 Tickets disponibles:")
    for k, v in TICKETS.items():
        preview = v.replace("\n", " ")[:80]
        print(f"  {k}. {preview}...")
    op = int(input("Elige ticket (1-4): ").strip() or "1")
    return TICKETS[op]


def elegir_rol():
    print("\n🎭 Roles disponibles:")
    for k, v in ROLES.items():
        print(f"  {k}. {v[:70]}...")
    op = int(input("Elige rol (1-3): ").strip() or "1")
    return ROLES[op]



In [ ]:

# ============================================================
#  OPCIÓN 1: LLAMADA BÁSICA
# ============================================================
def llamada_basica():
    rol = elegir_rol()
    ticket = elegir_ticket()
    temp = float(input("\n🌡  TEMPERATURE (0.0-2.0) [0.3]: ").strip() or "0.3")
    max_t = int(input("📏 MAX_TOKENS [400]: ").strip() or "400")

    print("\n⏳ Generando respuesta...\n")
    r = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": rol},
            {"role": "user", "content": ticket}
        ],
        temperature=temp,
        max_tokens=max_t,
    )
    print("=== RESPUESTA ===")
    print(r.choices[0].message.content)
    print(f"\n📊 Tokens usados: prompt={r.usage.prompt_tokens}, "
          f"respuesta={r.usage.completion_tokens}, total={r.usage.total_tokens}")



In [ ]:

# ============================================================
#  OPCIÓN 2: COMPARAR TEMPERATURES
# ============================================================
def comparar_temperatures():
    ticket = elegir_ticket()
    rol = "Eres un asistente que propone nombres creativos y soluciones."

    for temp in [0.0, 0.7, 1.5]:
        print(f"\n{'=' * 60}")
        print(f"  TEMPERATURE = {temp}")
        print("=" * 60)
        r = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": rol},
                {"role": "user", "content": ticket}
            ],
            temperature=temp,
            max_tokens=500,
        )
        print(r.choices[0].message.content)

    print("\n💡 Observa cómo a mayor temperature, más variada y creativa la respuesta.")



In [ ]:

# ============================================================
#  OPCIÓN 3: SALIDA ESTRUCTURADA (JSON)
# ============================================================
def salida_json():
    ticket = elegir_ticket()
    print("\n⏳ Clasificando en JSON...\n")
    r = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content": (
                    "Devuelves SIEMPRE un JSON válido con los campos: "
                    "categoria, urgencia (1-5), sentimiento, "
                    "requiere_humano (bool), resumen (max 30 palabras)."
                )
            },
            {"role": "user", "content": ticket}
        ],
        temperature=0,
        response_format={"type": "json_object"},
        max_tokens=300,
    )
    datos = json.loads(r.choices[0].message.content)
    print(json.dumps(datos, indent=2, ensure_ascii=False))
    print("\n💡 Este JSON ya se puede guardar en BD o mandar a un dashboard.")



In [ ]:

# ============================================================
#  OPCIÓN 4: CONVERSACIÓN MULTI-TURNO
# ============================================================
def multi_turno():
    print("\n💬 Modo conversación. Escribe 'salir' para terminar.\n")
    historial = [
        {"role": "system", "content": "Eres un asistente de RRHH amable y preciso."}
    ]
    # Pre-cargamos el primer turno como ejemplo
    primer_msg = "Hola, llevo 3 años en la empresa, ¿cuántos días de vacaciones tengo?"
    print(f"👤 Tú (ejemplo): {primer_msg}")
    historial.append({"role": "user", "content": primer_msg})

    r = client.chat.completions.create(
        model="deepseek-chat", messages=historial, temperature=0.5, max_tokens=200,
    )
    respuesta = r.choices[0].message.content
    historial.append({"role": "assistant", "content": respuesta})
    print(f"🤖 Bot: {respuesta}\n")

    # Ahora el alumno continúa la conversación
    while True:
        msg = input("👤 Tú: ").strip()
        if msg.lower() == "salir":
            break
        historial.append({"role": "user", "content": msg})
        r = client.chat.completions.create(
            model="deepseek-chat", messages=historial, temperature=0.5, max_tokens=200,
        )
        respuesta = r.choices[0].message.content
        historial.append({"role": "assistant", "content": respuesta})
        print(f"🤖 Bot: {respuesta}\n")

    print(f"📊 Turnos en historial: {len(historial)}")
    print("💡 El historial completo se reenvía en cada llamada (LLMs son stateless).")




In [ ]:
# ============================================================
#  OPCIÓN 5: STREAMING
# ============================================================
def streaming():
    pregunta = "Explícame qué es un LLM en 3 frases simples."
    print(f"\n👤 Pregunta: {pregunta}\n")
    print("🤖 Bot (streaming): ", end="", flush=True)

    stream = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": pregunta}],
        temperature=0.5,
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
    print("\n\n💡 El streaming mejora la UX: el usuario ve la respuesta aparecer.")




In [ ]:
# ============================================================
#  OPCIÓN 6: FUNCTION CALLING
# ============================================================
def function_calling():
    tools = [{
        "type": "function",
        "function": {
            "name": "consultar_estado_pedido",
            "description": "Consulta el estado de un pedido por su ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "pedido_id": {"type": "string", "description": "ID del pedido"}
                },
                "required": ["pedido_id"]
            }
        }
    }]
    consulta = "¿Dónde está mi pedido P-12345? Lo necesito hoy."
    print(f"\n👤 Cliente: {consulta}\n")

    r = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": consulta}],
        tools=tools,
        tool_choice="auto",
    )
    tool_calls = r.choices[0].message.tool_calls
    if tool_calls:
        for call in tool_calls:
            print("🔧 El modelo quiere llamar una función:")
            print(f"   Función   : {call.function.name}")
            print(f"   Argumentos: {call.function.arguments}")
        print("\n💡 En un sistema real, ejecutas la función y le devuelves "
              "el resultado al modelo en un nuevo turno.")
    else:
        print(r.choices[0].message.content)



In [ ]:
# ============================================================
#  OPCIÓN 7: ALUCINACIONES
# ============================================================
def alucinaciones():
    pregunta = "Dame el PBI exacto de Perú en el primer trimestre de 2026, citando fuente."
    print(f"\n👤 Pregunta difícil: {pregunta}\n")

    print("--- SIN mitigación ---")
    r1 = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": pregunta}],
        temperature=0.7,
    )
    print(r1.choices[0].message.content)

    print("\n--- CON mitigación ---")
    r2 = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content": (
                    "Responde SOLO con información que tengas certeza. "
                    "Si no estás seguro, di literalmente: "
                    "'No tengo información confiable sobre eso'. "
                    "No inventes cifras ni fuentes."
                )
            },
            {"role": "user", "content": pregunta}
        ],
        temperature=0,
    )
    print(r2.choices[0].message.content)
    print("\n💡 La instrucción explícita reduce alucinaciones.")


# ============================================================
#  LOOP PRINCIPAL
# ============================================================
acciones = {
    "1": llamada_basica,
    "2": comparar_temperatures,
    "3": salida_json,
    "4": multi_turno,
    "5": streaming,
    "6": function_calling,
    "7": alucinaciones,
}

while True:
    op = mostrar_menu()
    if op == "0":
        print("\n👋 ¡Hasta la próxima clase!")
        break
    if op in acciones:
        acciones[op]()
        input("\n⏎ Presiona Enter para volver al menú...")
    else:
        print("⚠ Opción inválida.")